# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [15]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [14]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Add uv to PATH for this notebook session
# import os
# os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]

# # Check uv works
# !uv --version

# # Create venv
# !uv venv .venv --seed

# # Install dependencies into the venv
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart kernel, then select: Python (cse151b)")

### Run the cell below every time to activate the installed environment. 

In [1]:
# # activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

In [1]:
# import sys
# print(sys.executable)

/workspace/151B_SP26_Competition/.venv/bin/python


In [13]:
# import sys, subprocess

# pkgs = ["torch", "transformers", "vllm", "bitsandbytes"]

# for pkg in pkgs:
#     print(f"\nTesting {pkg}...")
#     result = subprocess.run(
#         [sys.executable, "-c", f"import {pkg}; print('{pkg} OK')"],
#         capture_output=True,
#         text=True,
#         timeout=60,
#     )
#     print(result.stdout)
#     print(result.stderr)

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
print("Task started...")

import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

print("Task complete!")

Task started...
Task complete!


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
print("Task started...")

data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

print("Task complete!")

Task started...
Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}
Task complete!


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [16]:
# import sys

# !{sys.executable} -m pip uninstall -y vllm
# !{sys.executable} -m pip install "vllm==0.19.1"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

(EngineCore pid=2151) INFO 05-03 03:22:04 [weight_utils.py:581] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 20.987014 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=2151) INFO 05-03 03:22:06 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 37.523080 seconds
(EngineCore pid=2151) INFO 05-03 03:22:38 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/8c9f128ca0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2151) INFO 05-03 03:22:38 [backends.py:1111] Dynamo bytecode transform time: 30.80 s
(EngineCore pid=2151) INFO 05-03 03:22:47 [backends.py:372] Cache the graph of compile range (1, 32768) for later use
(EngineCore pid=2151) INFO 05-03 03:22:53 [backends.py:390] Compiling a graph for compile range (1, 32768) takes 15.23 s
(EngineCore pid=2151) INFO 05-03 03:23:06 [decorators.py:655] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/8c300eb6da3e0ead728b359a7a6320e1b3efb48edaff76c86977facba6772916/rank_0_0/model
(EngineCore pid=2151) INFO 05-03 03:23:06 [monitor.py:48] torch.compile took 58.83 s in total
(EngineCore pid=2151) INFO 05-03 03:

(EngineCore pid=2151) 2026-05-03 03:23:16,315 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=2151) 2026-05-03 03:23:16,408 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 10.85it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 14.66it/s]


(EngineCore pid=2151) INFO 05-03 03:23:24 [gpu_model_runner.py:6046] Graph capturing finished in 8 secs, took 0.70 GiB
(EngineCore pid=2151) INFO 05-03 03:23:24 [gpu_worker.py:597] CUDA graph pool memory: 0.7 GiB (actual), 0.49 GiB (estimated), difference: 0.21 GiB (29.3%).
(EngineCore pid=2151) INFO 05-03 03:23:24 [core.py:283] init engine (profile, create kv cache, warmup model) took 77.43 seconds
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [17]:
# import sys
# !{sys.executable} -m pip install -U accelerate

In [6]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [7]:
# Build prompts for first 5 entries MODIFIED TO ALL
prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 1126 questions...


Rendering prompts:   0%|          | 0/1126 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1126 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


── Response 0 (id=0) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to find the sum of the first 325 positive even whole numbers. Hmm, let's start by making sure I know what the first few even positive whole numbers are. Positive even whole numbers start at 2, right? So the first one is 2, second is 4, third is 6, fourth  ...

── Response 1 (id=1) ──
Okay, let's try to solve this integral: the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s² + a²) ds. Hmm, first, I need to check if the integral converges. Since the integrand is even (because s² is even, so the function is even in s), we can write it as 2 times the integral from 0 to infinity of (a^(3/2))/(s² + a²) ds. That might help.

But first, let's recall th ...

── Response 2 (id=2) ──
Okay, let's try to solve this problem step by step. First, part (a) is about the turkey cooling down, so I think we

### Generate with Transformers (for Datahub)

In [18]:
# from transformers import TextStreamer

# import torch
# print(torch.cuda.is_available(), "\n")
# print(torch.cuda.get_device_name(0), "\n")
# print(next(llm.parameters()).device, "\n")

# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [
#             {"role": "system", "content": system},
#             {"role": "user", "content": user},
#         ],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# responses = []

# for i, prompt in enumerate(prompts):
#     print(f"\n── Generating Response {i} (id={data[i].get('id')}) ──")

#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length=16384,
#     ).to(llm.device)

#     streamer = TextStreamer(
#         tokenizer,
#         skip_prompt=True,
#         skip_special_tokens=True,
#     )

#     with torch.no_grad():
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             repetition_penalty=1.0,
#             do_sample=True,
#             streamer=streamer,
#         )

#     # Decode only new tokens
#     new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
#     response = tokenizer.decode(
#         new_tokens,
#         skip_special_tokens=True,
#     ).strip()

#     responses.append(response)

#     print(f"\n── Finished Response {i} ──")

In [19]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.inference_mode():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         do_sample=True,
#         repetition_penalty=1.0,
#         pad_token_id=tokenizer.eos_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [8]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 1126/1126 [01:33<00:00, 12.07it/s]

Scoring complete. 1126 results.


## 8. Summary

Print accuracy broken down by question type.

In [9]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :  276 /  375  (73.60%)
  Free-form  :  416 /  751  (55.39%)
  Overall    :  692 / 1126  (61.46%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [10]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 1126 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [1]:
import sys
print(sys.executable)
# Expect "/workspace/151B_SP26_Competition/.venv/bin/python"

/workspace/.venv/bin/python


# My Model

In [2]:
print("Cell 1: imports, config, data, prompts...")

import os
import re
import sys
import gc
import json
import time
import shutil
import subprocess
from pathlib import Path
from typing import Optional, Dict, List, Tuple
from collections import Counter, defaultdict

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

# ── Main run config ─────────────────────────────────────────────
RUN_NAME = "vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100"

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = "data/public.jsonl"

# ── MCQ fallback adapter config ─────────────────────────────────
# Primary solver is always base model.
# Fallback is only used for MCQ items that base did not verify/save.
USE_MCQ_LORA_FALLBACK = True
FALLBACK_LORA_PATH = "outputs/qwen3_math_sft_r16_then_dpo_fixed_run1"
FALLBACK_LORA_NAME = "math_sft_r16_then_dpo_fixed_run1"
FALLBACK_LORA_RANK = 16

# If using r=4 LoRA, set rank to 8 for vLLM max_lora_rank compatibility.
# vLLM accepts max_lora_rank: 1, 8, 16, 32, 64, ...

# ── Result folder config ───────────────────────────────────────
RESULTS_ROOT = Path("results")
RUN_DIR = RESULTS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

RESPONSE_PATH = RUN_DIR / "responses.jsonl"
ATTEMPT_PATH = RUN_DIR / "attempts.jsonl"
VOTE_PATH = RUN_DIR / "votes.jsonl"
VERIFY_PATH = RUN_DIR / "verify.jsonl"
EVAL_PATH = RUN_DIR / "eval.jsonl"
SUMMARY_PATH = RUN_DIR / "summary.json"
CONFIG_PATH = RUN_DIR / "config.json"

# ── GPU config ─────────────────────────────────────────────────
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

# ── vLLM/model config ──────────────────────────────────────────
MAX_MODEL_LEN = 131072

GPU_MEMORY_UTILIZATION = 0.85
MAX_NUM_SEQS = 512
MAX_NUM_BATCHED_TOKENS = 65536

# ── Voting / retry / verification config ───────────────────────
NUM_VOTES = 3
AGREE_THRESHOLD = 2

DEFAULT_MAX_TOKENS = 32768

# Do not reduce token usage. Retry budgets only stay same or increase.
RETRY_MAX_TOKENS_BY_ROUND = {
    1: 65536,
    2: 65536,
    3: 65536,
}

MAX_RETRIES = 2
EXTRA_RETRIES_AFTER_VERIFY_FAIL = 1
MAX_TOTAL_RETRY_ROUNDS = MAX_RETRIES + EXTRA_RETRIES_AFTER_VERIFY_FAIL

VERIFY_MAX_TOKENS = 32768

BATCH_SIZE = 256
RETRY_BATCH_SIZE = 128
VERIFY_BATCH_SIZE = 128

# Missing/unanswered counts as wrong. Do not save best unverified answers.
SAVE_BEST_AFTER_MAX_RETRIES = False

# Limit for testing. Use None for full dataset.
TEST_LIMIT = 100

print(f"RUN_NAME                  = {RUN_NAME}")
print(f"RUN_DIR                   = {RUN_DIR}")
print(f"PRIMARY_SOLVER            = base")
print(f"USE_MCQ_LORA_FALLBACK     = {USE_MCQ_LORA_FALLBACK}")
print(f"FALLBACK_LORA_PATH        = {FALLBACK_LORA_PATH if USE_MCQ_LORA_FALLBACK else None}")
print(f"RESPONSE_PATH             = {RESPONSE_PATH}")
print(f"ATTEMPT_PATH              = {ATTEMPT_PATH}")
print(f"VOTE_PATH                 = {VOTE_PATH}")
print(f"VERIFY_PATH               = {VERIFY_PATH}")
print(f"EVAL_PATH                 = {EVAL_PATH}")
print(f"MAX_MODEL_LEN             = {MAX_MODEL_LEN}")
print(f"DEFAULT_MAX_TOKENS        = {DEFAULT_MAX_TOKENS}")
print(f"RETRY_MAX_TOKENS_BY_ROUND = {RETRY_MAX_TOKENS_BY_ROUND}")
print(f"VERIFY_MAX_TOKENS         = {VERIFY_MAX_TOKENS}")
print(f"NUM_VOTES                 = {NUM_VOTES}")
print(f"AGREE_THRESHOLD           = {AGREE_THRESHOLD}")
print(f"MAX_RETRIES               = {MAX_RETRIES}")
print(f"EXTRA_RETRIES_AFTER_VERIFY_FAIL = {EXTRA_RETRIES_AFTER_VERIFY_FAIL}")
print(f"BATCH_SIZE                = {BATCH_SIZE}")
print(f"RETRY_BATCH_SIZE          = {RETRY_BATCH_SIZE}")
print(f"VERIFY_BATCH_SIZE         = {VERIFY_BATCH_SIZE}")
print(f"GPU_MEMORY_UTILIZATION    = {GPU_MEMORY_UTILIZATION}")

# ── Load data ──────────────────────────────────────────────────
t0 = time.time()

with open(DATA_PATH, "r") as f:
    data = [json.loads(line) for line in f]

if TEST_LIMIT is not None:
    data = data[:TEST_LIMIT]

n_mcq = sum(bool(d.get("options")) for d in data)
n_frq = len(data) - n_mcq

print(f"Loaded {len(data)} questions: {n_mcq} MCQ, {n_frq} FRQ")
print(f"Data loading took {time.time() - t0:.2f}s")

data_by_id = {str(item["id"]): item for item in data}

# Save config snapshot
config_snapshot = {
    "run_name": RUN_NAME,
    "model_id": MODEL_ID,
    "data_path": DATA_PATH,
    "test_limit": TEST_LIMIT,
    "primary_solver": "base",
    "use_mcq_lora_fallback": USE_MCQ_LORA_FALLBACK,
    "fallback_lora_path": FALLBACK_LORA_PATH if USE_MCQ_LORA_FALLBACK else None,
    "fallback_lora_name": FALLBACK_LORA_NAME if USE_MCQ_LORA_FALLBACK else None,
    "fallback_lora_rank": FALLBACK_LORA_RANK if USE_MCQ_LORA_FALLBACK else None,
    "max_model_len": MAX_MODEL_LEN,
    "default_max_tokens": DEFAULT_MAX_TOKENS,
    "retry_max_tokens_by_round": RETRY_MAX_TOKENS_BY_ROUND,
    "verify_max_tokens": VERIFY_MAX_TOKENS,
    "num_votes": NUM_VOTES,
    "agree_threshold": AGREE_THRESHOLD,
    "max_retries": MAX_RETRIES,
    "extra_retries_after_verify_fail": EXTRA_RETRIES_AFTER_VERIFY_FAIL,
    "batch_size": BATCH_SIZE,
    "retry_batch_size": RETRY_BATCH_SIZE,
    "verify_batch_size": VERIFY_BATCH_SIZE,
    "save_best_after_max_retries": SAVE_BEST_AFTER_MAX_RETRIES,
}

with open(CONFIG_PATH, "w") as f:
    json.dump(config_snapshot, f, indent=2)

# ── Prompts ────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are an expert competition mathematician. "
    "Solve the problem carefully and efficiently. "
    "Check for traps, arithmetic errors, algebraic mistakes, domain restrictions, "
    "and whether the final answer matches the requested format. "
    "Do not round unless the problem explicitly asks for rounding. "
    "Prefer exact forms such as fractions, radicals, logarithms, and symbolic expressions. "
    "At the end, output exactly one final answer in \\boxed{...}. "
    "For multiple-choice questions, the box must contain only the chosen letter. "
    "For free-response questions with multiple parts, include all requested answers "
    "inside one box, separated by commas, preserving order. "
    "Do not write anything after the boxed answer."
)

RETRY_SYSTEM_PROMPT = (
    "You are an expert competition mathematician re-evaluating a problem after "
    "previous solution attempts disagreed, failed, or were rejected by verification. "
    "Use previous attempts as evidence, but do not blindly trust them. "
    "Identify likely mistakes, recompute carefully, and produce one final answer. "
    "Do not round unless explicitly requested. "
    "If the problem asks for multiple values, include every requested value in order. "
    "Your final line must be exactly one \\boxed{...}. "
    "Do not write anything after the boxed answer."
)

VERIFY_SYSTEM_PROMPT = (
    "You are a strict mathematical verifier. "
    "You are given an original problem and a candidate solution/final answer. "
    "Independently verify whether the candidate boxed answer is correct and complete. "
    "If the candidate answer is fully correct, output exactly \\boxed{YES}. "
    "If it is wrong, incomplete, incorrectly formatted, rounded when exact form is needed, "
    "or missing any requested part, output exactly \\boxed{NO}. "
    "Do not output anything after the boxed YES or NO."
)

def build_problem_text(item: dict) -> str:
    question = item["question"]

    if item.get("options"):
        labels = [chr(65 + i) for i in range(len(item["options"]))]
        opts_text = "\n".join(
            f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, item["options"])
        )
        return (
            f"{question}\n\n"
            f"Options:\n{opts_text}\n\n"
            "Select the single best option. "
            "Compare your result against all choices. "
            "End with exactly one boxed letter, e.g. \\boxed{C}."
        )

    return (
        f"{question}\n\n"
        "Solve carefully. If there are multiple parts, blanks, or requested values, "
        "answer all of them in order. "
        "Do not round unless explicitly requested. "
        "End with exactly one boxed final answer."
    )

def make_initial_prompt(item: dict) -> str:
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_problem_text(item)},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

def find_boxed_spans(text: str):
    spans = []
    start = 0
    text = str(text)

    while True:
        idx = text.find("\\boxed{", start)
        if idx < 0:
            break

        content_start = idx + len("\\boxed{")
        depth = 1
        i = content_start

        while i < len(text) and depth > 0:
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
            i += 1

        if depth == 0:
            content = text[content_start:i - 1]
            spans.append((idx, i, content))

        start = max(i, idx + 1)

    return spans

def extract_boxed(text: str) -> str:
    spans = find_boxed_spans(text)
    return spans[-1][2].strip() if spans else ""

def normalize_answer_key(ans: str, is_mcq: bool) -> str:
    ans = str(ans).strip()

    if is_mcq:
        m = re.search(r"[A-Za-z]", ans)
        return m.group(0).upper() if m else ""

    ans = ans.lower()
    ans = re.sub(r"\s+", "", ans)
    ans = ans.replace("\\left", "").replace("\\right", "")
    return ans

def extract_answer_key(text: str, is_mcq: bool) -> str:
    boxed = extract_boxed(text)

    if boxed:
        return normalize_answer_key(boxed, is_mcq)

    if is_mcq:
        m = re.search(r'"answer"\s*:\s*"([A-Za-z])"', text)
        if m:
            return m.group(1).upper()

        matches = re.findall(r"\b([A-Z])\b", text.upper())
        return matches[-1] if matches else ""

    return ""

def has_complete_answer(text: str, is_mcq: bool) -> bool:
    key = extract_answer_key(text, is_mcq)
    if not key:
        return False

    if is_mcq:
        return bool(re.fullmatch(r"[A-Z]", key))

    return bool(extract_boxed(text))

def parse_verify_decision(text: str) -> bool:
    boxed = extract_boxed(text).strip().upper()

    if boxed == "YES":
        return True
    if boxed == "NO":
        return False

    tail = text.strip().upper()[-200:]
    if "\\BOXED{YES}" in tail or ("YES" in tail and "NO" not in tail):
        return True

    return False

def format_eta(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.0f}s"
    if seconds < 3600:
        return f"{seconds / 60:.1f}m"
    return f"{seconds / 3600:.1f}h"

def truncate_text(s: str, max_chars: int = 1200) -> str:
    s = str(s).strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars] + "\n...[truncated]..."

print("Cell 1 complete.")

Cell 1: imports, config, data, prompts...
RUN_NAME                  = vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100
RUN_DIR                   = results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100
PRIMARY_SOLVER            = base
USE_MCQ_LORA_FALLBACK     = True
FALLBACK_LORA_PATH        = outputs/qwen3_math_sft_r16_then_dpo_fixed_run1
RESPONSE_PATH             = results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100/responses.jsonl
ATTEMPT_PATH              = results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100/attempts.jsonl
VOTE_PATH                 = results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100/votes.jsonl
VERIFY_PATH               = results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100/verify.jsonl
EVAL_PATH                 = results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100/eval.jsonl
MAX_MODEL_LEN             = 131072
DEFAULT_MAX_TOKENS        = 32768
RETRY_MAX_TOKENS_BY_ROUND = {1: 65536, 2: 65536, 3: 

In [3]:
print("Cell 2: loading tokenizer and vLLM model...")

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded in {time.time() - t0:.2f}s")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nCurrent nvidia-smi before model load:")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print("Could not run nvidia-smi:", e)

# ── Fallback LoRA setup ────────────────────────────────────────
fallback_lora_request = None

if USE_MCQ_LORA_FALLBACK:
    from vllm.lora.request import LoRARequest

    assert Path(FALLBACK_LORA_PATH).exists(), f"Fallback LoRA path not found: {FALLBACK_LORA_PATH}"

    fallback_lora_request = LoRARequest(
        FALLBACK_LORA_NAME,
        1,
        FALLBACK_LORA_PATH,
    )

    print(f"MCQ fallback LoRA enabled: {FALLBACK_LORA_PATH}")
else:
    print("MCQ fallback LoRA disabled.")

def vllm_supported_lora_rank(rank: int) -> int:
    supported = [1, 8, 16, 32, 64, 128, 256, 320, 512]
    for r in supported:
        if rank <= r:
            return r
    return 512

VLLM_MAX_LORA_RANK = vllm_supported_lora_rank(FALLBACK_LORA_RANK) if USE_MCQ_LORA_FALLBACK else 16
print("VLLM_MAX_LORA_RANK:", VLLM_MAX_LORA_RANK)

# ── Load vLLM model ────────────────────────────────────────────
t0 = time.time()

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=MAX_NUM_SEQS,
    max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,

    # Enable LoRA support only so fallback can use it.
    # Base generations still pass lora_request=None.
    enable_lora=USE_MCQ_LORA_FALLBACK,
    max_loras=1 if USE_MCQ_LORA_FALLBACK else 0,
    max_lora_rank=VLLM_MAX_LORA_RANK,
)

print(f"Model loaded in {(time.time() - t0) / 60:.2f} min.")
print("Primary solver: base")
print("MCQ fallback LoRA:", FALLBACK_LORA_PATH if USE_MCQ_LORA_FALLBACK else None)

print("\nCurrent nvidia-smi after model load:")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print("Could not run nvidia-smi:", e)

print("Cell 2 complete.")

Cell 2: loading tokenizer and vLLM model...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

(EngineCore pid=3166) INFO 05-31 03:38:08 [bitsandbytes_loader.py:786] Loading weights with BitsAndBytes quantization. May take a while ...
(EngineCore pid=3166) INFO 05-31 03:38:12 [weight_utils.py:581] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 3.493957 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.33it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.64it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.33it/s]
(EngineCore pid=3166) 


(EngineCore pid=3166) INFO 05-31 03:38:14 [punica_selector.py:20] Using PunicaWrapperGPU.
(EngineCore pid=3166) INFO 05-31 03:38:14 [gpu_model_runner.py:4820] Model loading took 2.78 GiB memory and 10.630220 seconds
(EngineCore pid=3166) INFO 05-31 03:38:35 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/5f062b57d9/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=3166) INFO 05-31 03:38:35 [backends.py:1111] Dynamo bytecode transform time: 20.12 s
(EngineCore pid=3166) INFO 05-31 03:38:45 [backends.py:372] Cache the graph of compile range (1, 65536) for later use
(EngineCore pid=3166) INFO 05-31 03:38:50 [backends.py:390] Compiling a graph for compile range (1, 65536) takes 14.59 s
(EngineCore pid=3166) INFO 05-31 03:38:57 [decorators.py:655] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/353632c17bb19ee188594ef40ceba9f08e7ed0b08be34ba61b746c31187aa9b4/rank_0_0/model
(EngineCore pid=3166) INFO 05-31 03:38:

(EngineCore pid=3166) 2026-05-31 03:39:10,083 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=3166) 2026-05-31 03:39:10,421 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 102/102 [00:17<00:00,  5.81it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 102/102 [00:09<00:00, 10.42it/s]


(EngineCore pid=3166) INFO 05-31 03:39:38 [gpu_model_runner.py:6046] Graph capturing finished in 28 secs, took 1.54 GiB
(EngineCore pid=3166) INFO 05-31 03:39:38 [gpu_worker.py:597] CUDA graph pool memory: 1.54 GiB (actual), 1.44 GiB (estimated), difference: 0.11 GiB (6.8%).
(EngineCore pid=3166) INFO 05-31 03:39:38 [core.py:283] init engine (profile, create kv cache, warmup model) took 84.14 seconds
(EngineCore pid=3166) INFO 05-31 03:39:40 [vllm.py:790] Asynchronous scheduling is enabled.
Model loaded in 3.03 min.
Primary solver: base
MCQ fallback LoRA: outputs/qwen3_math_sft_r16_then_dpo_fixed_run1

Current nvidia-smi after model load:
Sun May 31 03:39:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | B

In [4]:
print("Cell 3: base verified pipeline + MCQ SFT→DPO fallback...")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Using run folder:", RUN_DIR)
print("Primary solver: base")
print("MCQ fallback:", USE_MCQ_LORA_FALLBACK, FALLBACK_LORA_PATH if USE_MCQ_LORA_FALLBACK else None)

THINK_END_TOKEN_ID = 151668

def extract_qwen_final_content_from_vllm(completion, raw_text: str):
    token_ids = getattr(completion, "token_ids", None)

    if token_ids is not None:
        ids = list(token_ids)

        try:
            idx = len(ids) - ids[::-1].index(THINK_END_TOKEN_ID)
            final_ids = ids[idx:]
            final_text = tokenizer.decode(final_ids, skip_special_tokens=True).strip()
            if final_text:
                return final_text, True
        except ValueError:
            pass

    if "</think>" in raw_text:
        return raw_text.split("</think>")[-1].strip(), True

    return raw_text.strip(), False

def make_sampling_params(max_tokens: int, n: int) -> SamplingParams:
    return SamplingParams(
        n=n,
        max_tokens=max_tokens,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        presence_penalty=0.0,
        repetition_penalty=1.0,
    )

def make_verify_sampling_params() -> SamplingParams:
    return SamplingParams(
        n=1,
        max_tokens=VERIFY_MAX_TOKENS,
        temperature=0.0,
        top_p=1.0,
        top_k=-1,
        min_p=0.0,
        presence_penalty=0.0,
        repetition_penalty=1.0,
    )

def max_tokens_for_round(retry_num: int) -> int:
    if retry_num == 0:
        return DEFAULT_MAX_TOKENS
    return RETRY_MAX_TOKENS_BY_ROUND.get(
        retry_num,
        max(RETRY_MAX_TOKENS_BY_ROUND.values()),
    )

# ── Load checkpoint files ──────────────────────────────────────
responses_by_id = {}

if RESPONSE_PATH.exists():
    with open(RESPONSE_PATH, "r") as f:
        for line in f:
            row = json.loads(line)
            responses_by_id[str(row["id"])] = row

print(f"Already verified/accepted: {len(responses_by_id)}/{len(data)}")

attempts_by_id = defaultdict(list)

if ATTEMPT_PATH.exists():
    with open(ATTEMPT_PATH, "r") as f:
        for line in f:
            row = json.loads(line)
            attempts_by_id[str(row["id"])].append(row)

print(f"Loaded prior attempts for {len(attempts_by_id)} questions")

verify_by_id = defaultdict(list)

if VERIFY_PATH.exists():
    with open(VERIFY_PATH, "r") as f:
        for line in f:
            row = json.loads(line)
            verify_by_id[str(row["id"])].append(row)

print(f"Loaded prior verification records for {len(verify_by_id)} questions")

def save_responses_atomic():
    tmp_path = RESPONSE_PATH.with_suffix(".tmp")
    backup_path = RESPONSE_PATH.with_suffix(".bak")

    with open(tmp_path, "w") as f:
        for item in data:
            qid = str(item["id"])
            if qid in responses_by_id:
                f.write(json.dumps(responses_by_id[qid]) + "\n")

    if RESPONSE_PATH.exists():
        shutil.copy2(RESPONSE_PATH, backup_path)

    tmp_path.replace(RESPONSE_PATH)

def append_attempt_record(record: dict):
    with open(ATTEMPT_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

    attempts_by_id[str(record["id"])].append(record)

def append_vote_record(record: dict):
    with open(VOTE_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

def append_verify_record(record: dict):
    with open(VERIFY_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

    verify_by_id[str(record["id"])].append(record)

def summarize_attempts_for_retry(item: dict, solver_filter: Optional[str] = None, max_attempts: int = 18) -> str:
    qid = str(item["id"])
    attempts = attempts_by_id.get(qid, [])

    if solver_filter is not None:
        attempts = [a for a in attempts if a.get("solver") == solver_filter]

    attempts = attempts[-max_attempts:]

    lines = []

    all_attempts_for_counts = attempts_by_id.get(qid, [])
    counts = Counter(
        a.get("answer_key", "")
        for a in all_attempts_for_counts
        if a.get("answer_key")
    )

    if counts:
        lines.append("Candidate final answers from all previous attempts:")
        for ans, count in counts.most_common():
            lines.append(f"- {ans}: {count} vote(s)")
    else:
        lines.append("No previous attempt produced a parsable final answer.")

    if attempts:
        lines.append("\nPrevious reasoning/final-output snippets:")
        for i, a in enumerate(attempts, start=1):
            ans = a.get("answer_key", "")
            complete = a.get("complete", False)
            reached = a.get("reached_think_end", False)
            solver = a.get("solver", "unknown")
            final_text = truncate_text(a.get("final_text", ""), 700)
            raw_preview = truncate_text(a.get("raw_preview", ""), 700)

            lines.append(
                f"\nAttempt {i}: solver={solver}, answer_key={ans!r}, complete={complete}, reached_think_end={reached}\n"
                f"Final extracted text:\n{final_text if final_text else '[empty]'}\n"
                f"Raw reasoning/output preview:\n{raw_preview if raw_preview else '[empty]'}"
            )
    else:
        lines.append("\nNo previous attempts were recorded for this solver.")

    verifies = verify_by_id.get(qid, [])
    if verifies:
        lines.append("\nVerification results from previous selected answers:")
        for i, v in enumerate(verifies[-8:], start=1):
            lines.append(
                f"- Verification {i}: solver={v.get('solver')}, verified={v.get('verified')}, "
                f"candidate_answer={v.get('candidate_answer_key')!r}, "
                f"verifier_box={v.get('verifier_boxed')!r}"
            )
            verifier_text = truncate_text(v.get("verifier_final_text", ""), 500)
            if verifier_text:
                lines.append(f"  Verifier output: {verifier_text}")

    return "\n".join(lines)

def make_retry_prompt(item: dict, retry_num: int, max_tokens: int, solver_name: str) -> str:
    problem_text = build_problem_text(item)
    previous_summary = summarize_attempts_for_retry(item)

    user = (
        f"Original problem:\n{problem_text}\n\n"
        f"Previous generations disagreed, were incomplete, or failed verification.\n\n"
        f"{previous_summary}\n\n"
        f"You are now retrying with solver: {solver_name}.\n"
        "Re-evaluate the problem from scratch. "
        "Pay special attention to the conflicting candidate answers and verification failures above. "
        "Decide which answer is correct, or compute a new answer if all previous candidates are wrong. "
        "If the problem asks for multiple values, include every required value in order. "
        "Do not round unless explicitly requested. "
        "End with exactly one final answer in \\boxed{...}, and do not write anything after it."
    )

    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": RETRY_SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    max_prompt_tokens = max(1024, MAX_MODEL_LEN - max_tokens - 1024)
    ids = tokenizer.encode(prompt)

    if len(ids) > max_prompt_tokens:
        ids = ids[:max_prompt_tokens]
        prompt = tokenizer.decode(ids, skip_special_tokens=True)

    return prompt

def make_verify_prompt(item: dict, best_record: dict) -> str:
    problem_text = build_problem_text(item)
    candidate_answer = best_record.get("answer_key", "")
    candidate_solution = best_record.get("final_text", "")

    user = (
        f"Original problem:\n{problem_text}\n\n"
        f"Candidate boxed answer key/content:\n{candidate_answer}\n\n"
        f"Candidate solution/final output:\n{candidate_solution}\n\n"
        "Independently verify the candidate answer. "
        "Do not trust the candidate reasoning unless it checks out. "
        "If the boxed answer is correct and complete for the original problem, output exactly \\boxed{YES}. "
        "Otherwise output exactly \\boxed{NO}."
    )

    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": VERIFY_SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

def select_majority_response(item: dict, generation_records: List[dict]):
    complete_records = [
        r for r in generation_records
        if r["complete"] and r["answer_key"]
    ]

    if not complete_records:
        return None, {
            "accepted": False,
            "reason": "no_complete_records",
            "vote_counts": {},
            "top_answer": "",
            "top_count": 0,
        }

    counts = Counter(r["answer_key"] for r in complete_records)
    top_answer, top_count = counts.most_common(1)[0]

    if top_count >= AGREE_THRESHOLD:
        candidates = [r for r in complete_records if r["answer_key"] == top_answer]
        best = min(candidates, key=lambda r: len(r["final_text"]))

        return best, {
            "accepted": True,
            "reason": "majority_agreement",
            "vote_counts": dict(counts),
            "top_answer": top_answer,
            "top_count": top_count,
        }

    return None, {
        "accepted": False,
        "reason": "insufficient_agreement",
        "vote_counts": dict(counts),
        "top_answer": top_answer,
        "top_count": top_count,
    }

def save_verified_response(item: dict, best_record: dict, vote_info: dict, verify_record: dict, retry_num: int, max_tokens: int, solver_name: str):
    qid = str(item["id"])

    responses_by_id[qid] = {
        "id": item["id"],
        "is_mcq": bool(item.get("options")),
        "response": best_record["final_text"],
        "answer_key": best_record["answer_key"],
        "accepted_reason": "verified_majority",
        "vote_reason": vote_info["reason"],
        "verified": True,
        "verifier_boxed": verify_record.get("verifier_boxed", ""),
        "top_answer": vote_info["top_answer"],
        "top_count": vote_info["top_count"],
        "vote_counts": vote_info["vote_counts"],
        "num_votes": NUM_VOTES,
        "agree_threshold": AGREE_THRESHOLD,
        "retry_num": retry_num,
        "max_tokens": max_tokens,
        "solver": solver_name,
        "use_lora": solver_name != "base",
        "lora_path": FALLBACK_LORA_PATH if solver_name != "base" else None,
    }

def generate_votes_for_batch(batch: List[dict], retry_num: int, max_tokens: int, solver_name: str, lora_req):
    if retry_num == 0:
        prompts = [make_initial_prompt(item) for item in batch]
    else:
        prompts = [
            make_retry_prompt(item, retry_num=retry_num, max_tokens=max_tokens, solver_name=solver_name)
            for item in batch
        ]

    sampling_params = make_sampling_params(max_tokens=max_tokens, n=NUM_VOTES)

    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=lora_req,
    )

    batch_records = {}

    for item, out in zip(batch, outputs):
        qid = str(item["id"])
        is_mcq = bool(item.get("options"))

        records = []

        for vote_idx, completion in enumerate(out.outputs):
            raw_text = completion.text.strip()

            final_text, reached_think_end = extract_qwen_final_content_from_vllm(
                completion=completion,
                raw_text=raw_text,
            )

            answer_key = extract_answer_key(final_text, is_mcq)
            complete = has_complete_answer(final_text, is_mcq)

            record = {
                "id": item["id"],
                "is_mcq": is_mcq,
                "solver": solver_name,
                "retry_num": retry_num,
                "vote_idx": vote_idx,
                "max_tokens": max_tokens,
                "reached_think_end": reached_think_end,
                "complete": complete,
                "answer_key": answer_key,
                "final_text": final_text,
                "raw_preview": raw_text[:1200],
                "use_lora": solver_name != "base",
                "lora_path": FALLBACK_LORA_PATH if solver_name != "base" else None,
            }

            append_attempt_record(record)
            records.append(record)

        batch_records[qid] = records

    return batch_records

def verify_selected_batch(selected_items: List[Tuple[dict, dict, dict, int, int, str]], lora_req):
    """
    selected_items entries:
      (item, best_record, vote_info, retry_num, max_tokens, solver_name)
    """
    if not selected_items:
        return {}

    prompts = [
        make_verify_prompt(item, best_record)
        for item, best_record, vote_info, retry_num, max_tokens, solver_name in selected_items
    ]

    sampling_params = make_verify_sampling_params()

    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=lora_req,
    )

    verify_results = {}

    for (item, best_record, vote_info, retry_num, max_tokens, solver_name), out in zip(selected_items, outputs):
        qid = str(item["id"])
        completion = out.outputs[0]
        raw_text = completion.text.strip()

        final_text, reached_think_end = extract_qwen_final_content_from_vllm(
            completion=completion,
            raw_text=raw_text,
        )

        verifier_boxed = extract_boxed(final_text)
        verified = parse_verify_decision(final_text)

        record = {
            "id": item["id"],
            "is_mcq": bool(item.get("options")),
            "solver": solver_name,
            "retry_num": retry_num,
            "max_tokens": max_tokens,
            "candidate_answer_key": best_record.get("answer_key", ""),
            "candidate_final_text": best_record.get("final_text", ""),
            "vote_counts": vote_info.get("vote_counts", {}),
            "top_answer": vote_info.get("top_answer", ""),
            "top_count": vote_info.get("top_count", 0),
            "verified": verified,
            "verifier_boxed": verifier_boxed,
            "verifier_final_text": final_text,
            "verifier_raw_preview": raw_text[:1200],
            "reached_think_end": reached_think_end,
            "use_lora": solver_name != "base",
            "lora_path": FALLBACK_LORA_PATH if solver_name != "base" else None,
        }

        append_verify_record(record)
        verify_results[qid] = record

    return verify_results

def run_verified_solver_pipeline(items: List[dict], solver_name: str, lora_req):
    """
    Runs the full vote→retry→verify pipeline for a supplied item list.
    Saves verified results into responses_by_id.
    Returns unresolved items after all rounds.
    """
    total_start = time.time()

    unresolved = [
        item for item in items
        if str(item["id"]) not in responses_by_id
    ]

    print(f"\nStarting solver={solver_name}; unresolved count: {len(unresolved)}")

    for retry_num in range(MAX_TOTAL_RETRY_ROUNDS + 1):
        if not unresolved:
            break

        max_tokens = max_tokens_for_round(retry_num)
        if retry_num == 0:
            label = f"{solver_name}_initial"
        elif retry_num <= MAX_RETRIES:
            label = f"{solver_name}_retry_{retry_num}"
        else:
            label = f"{solver_name}_verify_fail_extra_retry_{retry_num}"

        print(f"\n=== {label}: {len(unresolved)} unresolved, max_tokens={max_tokens} ===")

        phase_start = time.time()
        next_unresolved = []

        phase_batch_size = BATCH_SIZE if retry_num == 0 else RETRY_BATCH_SIZE
        pbar = tqdm(range(0, len(unresolved), phase_batch_size), desc=label, dynamic_ncols=True)

        for batch_start in pbar:
            batch = unresolved[batch_start:batch_start + phase_batch_size]
            t0 = time.time()

            try:
                batch_records = generate_votes_for_batch(
                    batch=batch,
                    retry_num=retry_num,
                    max_tokens=max_tokens,
                    solver_name=solver_name,
                    lora_req=lora_req,
                )

                selected_for_verify = []
                no_majority_count = 0

                for item in batch:
                    qid = str(item["id"])
                    records = batch_records[qid]

                    best, vote_info = select_majority_response(item, records)

                    vote_record = {
                        "id": item["id"],
                        "is_mcq": bool(item.get("options")),
                        "solver": solver_name,
                        "retry_num": retry_num,
                        "max_tokens": max_tokens,
                        "selected_for_verification": best is not None,
                        "reason": vote_info["reason"],
                        "top_answer": vote_info["top_answer"],
                        "top_count": vote_info["top_count"],
                        "vote_counts": vote_info["vote_counts"],
                        "use_lora": solver_name != "base",
                        "lora_path": FALLBACK_LORA_PATH if solver_name != "base" else None,
                    }
                    append_vote_record(vote_record)

                    if best is not None:
                        selected_for_verify.append((item, best, vote_info, retry_num, max_tokens, solver_name))
                    else:
                        next_unresolved.append(item)
                        no_majority_count += 1

                verified_count = 0
                rejected_by_verify_count = 0

                for v_start in range(0, len(selected_for_verify), VERIFY_BATCH_SIZE):
                    verify_chunk = selected_for_verify[v_start:v_start + VERIFY_BATCH_SIZE]
                    verify_results = verify_selected_batch(verify_chunk, lora_req=lora_req)

                    for item, best, vote_info, rnum, mtoks, sname in verify_chunk:
                        qid = str(item["id"])
                        verify_record = verify_results[qid]

                        if verify_record["verified"]:
                            save_verified_response(
                                item=item,
                                best_record=best,
                                vote_info=vote_info,
                                verify_record=verify_record,
                                retry_num=rnum,
                                max_tokens=mtoks,
                                solver_name=sname,
                            )
                            verified_count += 1
                        else:
                            next_unresolved.append(item)
                            rejected_by_verify_count += 1

                save_responses_atomic()

                batch_time = time.time() - t0
                elapsed = time.time() - phase_start
                processed = min(batch_start + len(batch), len(unresolved))
                avg = elapsed / max(processed, 1)
                eta = avg * (len(unresolved) - processed)

                pbar.set_postfix({
                    "verified_total": f"{len(responses_by_id)}/{len(data)}",
                    "verified_batch": verified_count,
                    "no_maj": no_majority_count,
                    "verif_no": rejected_by_verify_count,
                    "batch_s": f"{batch_time:.1f}",
                    "ETA": format_eta(eta),
                })

            except Exception as e:
                print("\nGeneration or verification crashed. Saving accepted responses before raising error...")
                save_responses_atomic()
                print(f"Saved {len(responses_by_id)} verified responses to {RESPONSE_PATH}")
                raise e

            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        unresolved = next_unresolved

        print(
            f"{label} complete. Verified/accepted so far: {len(responses_by_id)}/{len(data)}. "
            f"Still unresolved for this solver: {len(unresolved)}. "
            f"Phase time: {format_eta(time.time() - phase_start)}"
        )

    if unresolved:
        print(f"\nSolver={solver_name} leaving {len(unresolved)} unresolved/unverified items.")

    print(f"Solver={solver_name} total time: {format_eta(time.time() - total_start)}")
    return unresolved

# ── Stage 1: Base for all questions ─────────────────────────────
base_unresolved = run_verified_solver_pipeline(
    items=data,
    solver_name="base",
    lora_req=None,
)

# ── Stage 2: SFT→DPO fallback only for unresolved MCQs ──────────
fallback_unresolved = []
if USE_MCQ_LORA_FALLBACK:
    mcq_fallback_items = [
        item for item in base_unresolved
        if bool(item.get("options")) and str(item["id"]) not in responses_by_id
    ]

    print(f"\nMCQ fallback candidates after base: {len(mcq_fallback_items)}")

    if mcq_fallback_items:
        fallback_unresolved = run_verified_solver_pipeline(
            items=mcq_fallback_items,
            solver_name="mcq_sftdpo_fallback",
            lora_req=fallback_lora_request,
        )
else:
    print("\nMCQ fallback disabled.")

# ── Final unresolved summary ───────────────────────────────────
all_missing = [
    item for item in data
    if str(item["id"]) not in responses_by_id
]

missing_mcq = [item["id"] for item in all_missing if item.get("options")]
missing_frq = [item["id"] for item in all_missing if not item.get("options")]

print("\nGeneration complete.")
print(f"Verified/saved responses: {len(responses_by_id)}/{len(data)}")
print(f"Missing total: {len(all_missing)}")
print(f"Missing MCQ  : {len(missing_mcq)} {missing_mcq[:20]}")
print(f"Missing FRQ  : {len(missing_frq)} {missing_frq[:20]}")
print(f"Responses saved to: {RESPONSE_PATH}")
print(f"Attempts saved to: {ATTEMPT_PATH}")
print(f"Vote records saved to: {VOTE_PATH}")
print(f"Verification records saved to: {VERIFY_PATH}")

save_responses_atomic()

responses = [
    responses_by_id[str(item["id"])]["response"]
    for item in data
    if str(item["id"]) in responses_by_id
]

response_ids = [
    item["id"]
    for item in data
    if str(item["id"]) in responses_by_id
]

print("Cell 3 complete.")

Cell 3: base verified pipeline + MCQ SFT→DPO fallback...
Using run folder: results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100
Primary solver: base
MCQ fallback: True outputs/qwen3_math_sft_r16_then_dpo_fixed_run1
Already verified/accepted: 0/100
Loaded prior attempts for 0 questions
Loaded prior verification records for 0 questions

Starting solver=base; unresolved count: 100

=== base_initial: 100 unresolved, max_tokens=32768 ===


base_initial:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/93 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

mcq_sftdpo_fallback_verify_fail_extra_retry_3 complete. Verified/accepted so far: 97/100. Still unresolved for this solver: 3. Phase time: 15.9m

Solver=mcq_sftdpo_fallback leaving 3 unresolved/unverified items.
Solver=mcq_sftdpo_fallback total time: 1.3h

Generation complete.
Verified/saved responses: 97/100
Missing total: 3
Missing MCQ  : 3 [1, 95, 99]
Missing FRQ  : 0 []
Responses saved to: results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100/responses.jsonl
Attempts saved to: results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100/attempts.jsonl
Vote records saved to: results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100/votes.jsonl
Verification records saved to: results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100/verify.jsonl
Cell 3 complete.


In [5]:
print("Cell 4: evaluation and score saving...")

t0 = time.time()

responses_by_id = {}

if RESPONSE_PATH.exists():
    with open(RESPONSE_PATH, "r") as f:
        for line in f:
            row = json.loads(line)
            responses_by_id[str(row["id"])] = row

print(f"Loaded {len(responses_by_id)}/{len(data)} verified responses from {RESPONSE_PATH}")

missing = [item["id"] for item in data if str(item["id"]) not in responses_by_id]
if missing:
    print(f"Warning: {len(missing)} responses missing/ignored. First few missing ids: {missing[:10]}")

# ── Judger setup ───────────────────────────────────────────────
sys.path.insert(0, ".")
from judger import Judger

judger = Judger(strict_extract=False)

def score_mcq(response: str, gold_letter: str) -> bool:
    pred = extract_answer_key(response, is_mcq=True)
    return pred == gold_letter.strip().upper()

# ── Evaluate answered responses ────────────────────────────────
results = []

eval_start = time.time()

for item in tqdm(data, desc="Scoring answered", dynamic_ncols=True):
    qid = str(item["id"])
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if qid not in responses_by_id:
        results.append({
            "id": item["id"],
            "is_mcq": is_mcq,
            "gold": gold,
            "response": "",
            "answer_key": "",
            "accepted_reason": "missing_unanswered",
            "vote_reason": "",
            "verified": False,
            "verifier_boxed": "",
            "top_count": None,
            "vote_counts": {},
            "has_boxed": False,
            "correct": False,
            "missing": True,
            "solver": None,
            "use_lora": False,
            "lora_path": None,
        })
        continue

    record = responses_by_id[qid]
    response = record["response"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]

        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id": item["id"],
        "is_mcq": is_mcq,
        "gold": gold,
        "response": response,
        "answer_key": record.get("answer_key", ""),
        "accepted_reason": record.get("accepted_reason", ""),
        "vote_reason": record.get("vote_reason", ""),
        "verified": record.get("verified", False),
        "verifier_boxed": record.get("verifier_boxed", ""),
        "top_count": record.get("top_count", None),
        "vote_counts": record.get("vote_counts", {}),
        "has_boxed": bool(extract_boxed(response)),
        "correct": correct,
        "missing": False,
        "solver": record.get("solver", "unknown"),
        "use_lora": record.get("use_lora", False),
        "lora_path": record.get("lora_path", None),
    })

eval_time = time.time() - eval_start

# ── Save eval results ──────────────────────────────────────────
with open(EVAL_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

# ── Summary ────────────────────────────────────────────────────
answered_results = [r for r in results if not r["missing"]]
mcq_all = [r for r in results if r["is_mcq"]]
frq_all = [r for r in results if not r["is_mcq"]]
mcq_answered = [r for r in answered_results if r["is_mcq"]]
frq_answered = [r for r in answered_results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

solver_counts = Counter(r["solver"] for r in answered_results)
solver_correct = Counter(r["solver"] for r in answered_results if r["correct"])

summary = {
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "primary_solver": "base",
    "use_mcq_lora_fallback": USE_MCQ_LORA_FALLBACK,
    "fallback_lora_path": FALLBACK_LORA_PATH if USE_MCQ_LORA_FALLBACK else None,
    "response_path": str(RESPONSE_PATH),
    "attempt_path": str(ATTEMPT_PATH),
    "vote_path": str(VOTE_PATH),
    "verify_path": str(VERIFY_PATH),
    "eval_path": str(EVAL_PATH),

    "num_total_data": len(data),
    "num_evaluated_answered_only": len(answered_results),
    "num_missing": sum(r["missing"] for r in results),

    # Accuracy over answered only.
    "mcq_correct_answered_only": sum(r["correct"] for r in mcq_answered),
    "mcq_total_answered_only": len(mcq_answered),
    "mcq_accuracy_answered_only": acc(mcq_answered),
    "frq_correct_answered_only": sum(r["correct"] for r in frq_answered),
    "frq_total_answered_only": len(frq_answered),
    "frq_accuracy_answered_only": acc(frq_answered),
    "overall_correct_answered_only": sum(r["correct"] for r in answered_results),
    "overall_total_answered_only": len(answered_results),
    "overall_accuracy_answered_only": acc(answered_results),

    # Accuracy over full dataset, missing/unanswered wrong.
    "mcq_correct_total": sum(r["correct"] for r in mcq_all),
    "mcq_total": len(mcq_all),
    "mcq_accuracy_total": acc(mcq_all),
    "frq_correct_total": sum(r["correct"] for r in frq_all),
    "frq_total": len(frq_all),
    "frq_accuracy_total": acc(frq_all),
    "overall_correct_total": sum(r["correct"] for r in results),
    "overall_total": len(results),
    "overall_accuracy_total": acc(results),

    # Backward-compatible alias.
    "overall_accuracy_missing_wrong": acc(results),

    "num_with_boxed": sum(r["has_boxed"] for r in answered_results),
    "num_without_boxed": sum(not r["has_boxed"] for r in answered_results),
    "num_verified": sum(bool(r["verified"]) for r in answered_results),

    "solver_counts": dict(solver_counts),
    "solver_correct": dict(solver_correct),

    "eval_time_sec": eval_time,
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)
print(f"RUN_NAME  : {RUN_NAME}")
print(f"RUN_DIR   : {RUN_DIR}")
print("-" * 70)
print("Answered-only accuracy:")
print(f"MCQ       : {summary['mcq_correct_answered_only']:4d} / {summary['mcq_total_answered_only']:4d} = {summary['mcq_accuracy_answered_only']:.2f}%")
print(f"FRQ       : {summary['frq_correct_answered_only']:4d} / {summary['frq_total_answered_only']:4d} = {summary['frq_accuracy_answered_only']:.2f}%")
print(f"Overall   : {summary['overall_correct_answered_only']:4d} / {summary['overall_total_answered_only']:4d} = {summary['overall_accuracy_answered_only']:.2f}%")
print("-" * 70)
print("Total accuracy, missing/unanswered wrong:")
print(f"MCQ       : {summary['mcq_correct_total']:4d} / {summary['mcq_total']:4d} = {summary['mcq_accuracy_total']:.2f}%")
print(f"FRQ       : {summary['frq_correct_total']:4d} / {summary['frq_total']:4d} = {summary['frq_accuracy_total']:.2f}%")
print(f"Overall   : {summary['overall_correct_total']:4d} / {summary['overall_total']:4d} = {summary['overall_accuracy_total']:.2f}%")
print("-" * 70)
print(f"Verified responses    : {summary['num_verified']}")
print(f"Missing/ignored       : {summary['num_missing']}")
print(f"With boxed answer     : {summary['num_with_boxed']}")
print(f"Without boxed answer  : {summary['num_without_boxed']}")
print("-" * 70)
print("Solver counts:", summary["solver_counts"])
print("Solver correct:", summary["solver_correct"])
print("-" * 70)
print(f"Saved eval to    : {EVAL_PATH}")
print(f"Saved summary to : {SUMMARY_PATH}")
print(f"Eval time        : {eval_time:.2f}s")
print("=" * 70)

print("Cell 4 complete.")

Cell 4: evaluation and score saving...
Loaded 97/100 verified responses from results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100/responses.jsonl


Scoring answered:   0%|          | 0/100 [00:00<?, ?it/s]

EVALUATION RESULTS
RUN_NAME  : vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100
RUN_DIR   : results/vllm_vote_verify_base_then_mcq_sftdpo_fallback_test100
----------------------------------------------------------------------
Answered-only accuracy:
MCQ       :   31 /   35 = 88.57%
FRQ       :   37 /   62 = 59.68%
Overall   :   68 /   97 = 70.10%
----------------------------------------------------------------------
Total accuracy, missing/unanswered wrong:
MCQ       :   31 /   38 = 81.58%
FRQ       :   37 /   62 = 59.68%
Overall   :   68 /  100 = 68.00%
----------------------------------------------------------------------
Verified responses    : 97
Missing/ignored       : 3
With boxed answer     : 97
Without boxed answer  : 0
----------------------------------------------------------------------
Solver counts: {'base': 96, 'mcq_sftdpo_fallback': 1}
Solver correct: {'base': 67, 'mcq_sftdpo_fallback': 1}
----------------------------------------------------------------------
Saved

[rank0]:[W531 07:16:36.142339652 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


ERROR 05-31 07:16:36 [core_client.py:667] Engine core proc EngineCore died unexpectedly, shutting down client.


# Potential Ensemble Method Gain

In [5]:
import json
from pathlib import Path
from collections import Counter

sys.path.insert(0, ".")
from judger import Judger

judger = Judger(strict_extract=False)

DATA_PATH = "data/public.jsonl"

runs = {
    "base_verify": "results/vllm_vote_verify_base_run1_responses.jsonl",
    "dpo_fixed": "results/vllm_vote_dpo_fixed_only_run1_test100_responses.jsonl",
    "sft_r16": "results/vllm_vote_lora_distill_run1_r16_responses.jsonl",
}

with open(DATA_PATH, "r") as f:
    data = [json.loads(line) for line in f][:100]

data_by_id = {str(item["id"]): item for item in data}

def load_responses(path):
    rows = {}
    if not Path(path).exists():
        print("Missing:", path)
        return rows

    with open(path, "r") as f:
        for line in f:
            row = json.loads(line)
            rows[str(row["id"])] = row
    return rows

responses = {
    name: load_responses(path)
    for name, path in runs.items()
}

def extract_boxed(text: str) -> str:
    import re
    matches = re.findall(r"\\boxed\{([^{}]*)\}", text, flags=re.DOTALL)
    return matches[-1].strip() if matches else ""

def extract_answer_key(text: str, is_mcq: bool) -> str:
    boxed = extract_boxed(text)

    if is_mcq:
        if boxed:
            import re
            m = re.search(r"[A-Za-z]", boxed)
            return m.group(0).upper() if m else ""

        import re
        matches = re.findall(r"\b([A-Z])\b", text.upper())
        return matches[-1] if matches else ""

    return boxed

def score_response(row, item):
    if row is None:
        return False

    response = row["response"]
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if is_mcq:
        pred = extract_answer_key(response, is_mcq=True)
        return pred == str(gold).strip().upper()

    gold_list = gold if isinstance(gold, list) else [gold]

    try:
        return bool(judger.auto_judge(
            pred=response,
            gold=gold_list,
            options=[[]] * len(gold_list),
        ))
    except Exception:
        return False

correct = {}

for run_name, rows in responses.items():
    correct[run_name] = {}

    for qid, item in data_by_id.items():
        correct[run_name][qid] = score_response(rows.get(qid), item)

    total = sum(correct[run_name].values())
    evaluated = sum(qid in rows for qid in data_by_id)

    print(f"{run_name:15s}: correct={total:3d}/100, evaluated={evaluated}")

all_qids = sorted(data_by_id.keys(), key=int)

oracle_correct = sum(
    any(correct[run_name][qid] for run_name in runs)
    for qid in all_qids
)

base_correct = sum(correct["base_verify"].values())

print()
print("Base correct:", base_correct)
print("Oracle ensemble correct:", oracle_correct)
print("Potential ensemble gain:", oracle_correct - base_correct)

print()
for run_name in runs:
    fixes = [
        qid for qid in all_qids
        if not correct["base_verify"][qid] and correct[run_name][qid]
    ]

    loses = [
        qid for qid in all_qids
        if correct["base_verify"][qid] and not correct[run_name][qid]
    ]

    print(f"{run_name}:")
    print("  fixes base misses:", len(fixes), fixes)
    print("  loses base wins  :", len(loses), loses)

print()
different_answer_ids = []

for qid in all_qids:
    keys = {}

    for run_name, rows in responses.items():
        row = rows.get(qid)
        keys[run_name] = row.get("answer_key") if row else None

    non_null = [v for v in keys.values() if v is not None]

    if len(set(non_null)) > 1:
        different_answer_ids.append(qid)

print("Questions with different answer keys:", len(different_answer_ids))
print(different_answer_ids)

print()
print("Detailed disagreement table:")
for qid in different_answer_ids:
    item = data_by_id[qid]
    print("=" * 80)
    print("id:", qid)
    print("gold:", item["answer"])
    for run_name, rows in responses.items():
        row = rows.get(qid)
        ans = row.get("answer_key") if row else None
        c = correct[run_name][qid]
        print(f"{run_name:15s}: correct={c}, answer_key={ans}")

base_verify    : correct= 67/100, evaluated=97
dpo_fixed      : correct= 60/100, evaluated=85
sft_r16        : correct= 61/100, evaluated=92

Base correct: 67
Oracle ensemble correct: 70
Potential ensemble gain: 3

base_verify:
  fixes base misses: 0 []
  loses base wins  : 0 []
dpo_fixed:
  fixes base misses: 1 ['82']
  loses base wins  : 8 ['3', '7', '29', '32', '65', '70', '75', '79']
sft_r16:
  fixes base misses: 3 ['82', '95', '99']
  loses base wins  : 9 ['7', '29', '32', '37', '49', '69', '70', '75', '79']

Questions with different answer keys: 29
['1', '2', '3', '5', '12', '16', '17', '21', '22', '27', '36', '37', '43', '44', '49', '57', '61', '62', '65', '66', '69', '75', '77', '82', '84', '86', '91', '95', '99']

Detailed disagreement table:
id: 1
gold: F
base_verify    : correct=False, answer_key=None
dpo_fixed      : correct=False, answer_key=E
sft_r16        : correct=False, answer_key=A
id: 2
gold: ['143.224229233795', '2.32624773420025']
base_verify    : correct=False, a

# LoRA + DPO

In [1]:
import time
import torch
from transformers import TrainerCallback

class TimerMemoryCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print("Training started.")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step == 0:
            return

        elapsed = time.time() - self.start_time
        step = state.global_step
        max_steps = state.max_steps
        sec_per_step = elapsed / max(step, 1)
        eta = (max_steps - step) * sec_per_step if max_steps and max_steps > 0 else 0

        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            max_allocated = torch.cuda.max_memory_allocated() / 1024**3
        else:
            allocated = reserved = max_allocated = 0

        print(
            f"[step {step}/{max_steps}] "
            f"elapsed={elapsed/60:.1f}m "
            f"eta={eta/60:.1f}m "
            f"sec/step={sec_per_step:.1f} "
            f"loss={logs.get('loss', 'n/a') if logs else 'n/a'} "
            f"alloc={allocated:.1f}GB "
            f"reserved={reserved:.1f}GB "
            f"max_alloc={max_allocated:.1f}GB"
        )

In [2]:
import gc
import time
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

TRAIN_DATA_PATH = "data/processed/distill_r1_qwen_math_clean_qwen_chat_no_token_filter"
SFT_OUTPUT_DIR = "outputs/qwen3_math_sft_r16_for_dpo_stage1"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()

train_ds = load_from_disk(TRAIN_DATA_PATH)

print(train_ds)
print("Train examples:", len(train_ds))
print("Max tokens:", max(train_ds["num_tokens"]))
print("Avg tokens:", sum(train_ds["num_tokens"]) / len(train_ds))
print("Example preview:")
print(train_ds[0]["text"][:1200])

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

sft_config = SFTConfig(
    output_dir=SFT_OUTPUT_DIR,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    num_train_epochs=0.5,

    logging_steps=10,
    save_steps=500,

    bf16=True,
    max_length=8192,
    packing=False,
    dataset_text_field="text",
    report_to="none",

    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
    callbacks=[TimerMemoryCallback()],
)

print("Starting SFT stage...")
trainer.train()
trainer.save_model(SFT_OUTPUT_DIR)

print(f"Saved SFT adapter to {SFT_OUTPUT_DIR}")
print(f"Total SFT wall time: {(time.time() - t0) / 60:.1f}m")

Dataset({
    features: ['file', 'answer', 'clean_status', 'clean_reason', 'text', 'num_tokens'],
    num_rows: 5405
})
Train examples: 5405
Max tokens: 5318
Avg tokens: 2487.377613320999
Example preview:
<|im_start|>system
You are an expert competition mathematician. Solve the problem carefully and efficiently. Do not round unless explicitly requested. End with exactly one final answer in \boxed{...}. Do not write anything after the boxed answer.<|im_end|>
<|im_start|>user
Define a sequence of complex numbers by $z_1 = 0$ and
\[z_{n + 1} = z_n^2 + i\]for all $n \ge 1.$  In the complex plane, how far from the origin is $z_{111}$?<|im_end|>
<|im_start|>assistant
<think>

</think>

To determine how far the point \( z_{111} \) is from the origin in the complex plane, we analyze the sequence defined by \( z_1 = 0 \) and \( z_{n + 1} = z_n^2 + i \).

1. **Initial Terms**:
 - \( z_1 = 0 \)
 - \( z_2 = z_1^2 + i = 0 + i = i \)
 - \( z_3 = z_2^2 + i = (-1) + i = -1 + i \)
 - \( z_4 = z_3^2 + i

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting SFT stage...
Training started.


Step,Training Loss
10,0.706639
20,0.707944
30,0.662150
40,0.608240
50,0.614321
60,0.561481
70,0.578342
80,0.606202
90,0.525935
100,0.528514


[step 10/338] elapsed=0.7m eta=22.9m sec/step=4.2 loss=0.7066388607025147 alloc=2.6GB reserved=33.5GB max_alloc=11.6GB
[step 20/338] elapsed=1.3m eta=21.1m sec/step=4.0 loss=0.7079435348510742 alloc=2.6GB reserved=33.5GB max_alloc=11.6GB
[step 30/338] elapsed=1.9m eta=19.9m sec/step=3.9 loss=0.6621498107910156 alloc=2.6GB reserved=33.5GB max_alloc=11.6GB
[step 40/338] elapsed=2.6m eta=19.1m sec/step=3.8 loss=0.6082395553588867 alloc=2.6GB reserved=33.5GB max_alloc=11.6GB
[step 50/338] elapsed=3.2m eta=18.3m sec/step=3.8 loss=0.6143211841583252 alloc=2.6GB reserved=33.5GB max_alloc=11.6GB
[step 60/338] elapsed=3.8m eta=17.6m sec/step=3.8 loss=0.5614811897277832 alloc=2.6GB reserved=33.5GB max_alloc=11.6GB
[step 70/338] elapsed=4.5m eta=17.0m sec/step=3.8 loss=0.5783424854278565 alloc=2.6GB reserved=41.8GB max_alloc=11.9GB
[step 80/338] elapsed=5.0m eta=16.2m sec/step=3.8 loss=0.6062016963958741 alloc=2.6GB reserved=41.8GB max_alloc=11.9GB
[step 90/338] elapsed=5.7m eta=15.6m sec/step=3.

In [3]:
import gc
import time
import inspect
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from trl import DPOTrainer, DPOConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

SFT_ADAPTER_PATH = "outputs/qwen3_math_sft_r16_for_dpo_stage1"
DPO_DATA_PATH = "data/processed/distill_r1_qwen_math_dpo_pairs_fixed_only"

FINAL_OUTPUT_DIR = "outputs/qwen3_math_sft_r16_then_dpo_fixed_run1"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()

dpo_ds = load_from_disk(DPO_DATA_PATH)

print(dpo_ds)
print("DPO pairs:", len(dpo_ds))
print("Example prompt:")
print(dpo_ds[0]["prompt"][:1000])
print("Example chosen:")
print(dpo_ds[0]["chosen"][:1000])
print("Example rejected:")
print(dpo_ds[0]["rejected"][:1000])

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

print("Loading SFT adapter as trainable...")
model = PeftModel.from_pretrained(
    model,
    SFT_ADAPTER_PATH,
    is_trainable=True,
)

model.print_trainable_parameters()

wanted_dpo_args = dict(
    output_dir=FINAL_OUTPUT_DIR,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    # Conservative because adapter already had SFT.
    learning_rate=3e-7,
    num_train_epochs=0.3,

    beta=0.1,

    logging_steps=10,
    save_steps=500,

    bf16=True,
    report_to="none",

    # These may or may not exist depending on TRL version.
    max_prompt_length=4096,
    max_length=8192,

    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

sig = inspect.signature(DPOConfig)

supported_args = {
    k: v for k, v in wanted_dpo_args.items()
    if k in sig.parameters
}

unsupported_args = {
    k: v for k, v in wanted_dpo_args.items()
    if k not in sig.parameters
}

print("\nUsing supported DPOConfig args:")
for k in supported_args:
    print(" ", k)

print("\nSkipped unsupported DPOConfig args:")
for k in unsupported_args:
    print(" ", k)

dpo_config = DPOConfig(**supported_args)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_ds,
    processing_class=tokenizer,
    callbacks=[TimerMemoryCallback()],
)

print("Starting DPO stage on top of SFT adapter...")
trainer.train()
trainer.save_model(FINAL_OUTPUT_DIR)

print(f"Saved final SFT→DPO adapter to {FINAL_OUTPUT_DIR}")
print(f"Total DPO wall time: {(time.time() - t0) / 60:.1f}m")

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'file', 'sample_idx_rejected', 'answer', 'chosen_clean_status', 'chosen_clean_reason', 'rejected_reason', 'prompt_tokens', 'chosen_tokens', 'rejected_tokens'],
    num_rows: 5405
})
DPO pairs: 5405
Example prompt:
<|im_start|>system
You are an expert competition mathematician. Solve the problem carefully and efficiently. Do not round unless explicitly requested. End with exactly one final answer in \boxed{...}. Do not write anything after the boxed answer.<|im_end|>
<|im_start|>user
Ben rolls 5 fair 12-sided dice.  The 12 faces of each die are numbered from 1 to 12. What is the probability that exactly two of the dice show an even number?<|im_end|>
<|im_start|>assistant
<think>

Example chosen:
... ... ...

Wait, the answer is missing.

Okay, so I'm going to try to figure this out. Let's start by understanding the problem.

Ben rolls 5 fair 12-sided dice. Each die has faces numbered from 1 to 12. We need to find the probability t

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading SFT adapter as trainable...
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145

Using supported DPOConfig args:
  output_dir
  per_device_train_batch_size
  gradient_accumulation_steps
  learning_rate
  num_train_epochs
  beta
  logging_steps
  save_steps
  bf16
  report_to
  max_length
  gradient_checkpointing
  optim

Skipped unsupported DPOConfig args:
  max_prompt_length


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting DPO stage on top of SFT adapter...
Training started.


Step,Training Loss
10,0.691540
20,0.717720
30,0.689740
40,0.689891
50,0.684672
60,0.697948
70,0.696213
80,0.695491
90,0.680578
100,0.671817


[step 10/203] elapsed=1.8m eta=35.3m sec/step=11.0 loss=0.691539716720581 alloc=3.5GB reserved=37.3GB max_alloc=27.7GB
[step 20/203] elapsed=3.6m eta=32.5m sec/step=10.7 loss=0.7177197456359863 alloc=3.5GB reserved=57.5GB max_alloc=28.0GB
[step 30/203] elapsed=5.3m eta=30.6m sec/step=10.6 loss=0.6897400379180908 alloc=3.5GB reserved=57.5GB max_alloc=28.0GB
[step 40/203] elapsed=7.0m eta=28.6m sec/step=10.5 loss=0.689891004562378 alloc=3.5GB reserved=78.4GB max_alloc=28.7GB
[step 50/203] elapsed=8.7m eta=26.5m sec/step=10.4 loss=0.6846719741821289 alloc=3.5GB reserved=35.1GB max_alloc=31.9GB
[step 60/203] elapsed=10.1m eta=24.1m sec/step=10.1 loss=0.6979477882385254 alloc=3.5GB reserved=35.2GB max_alloc=31.9GB
[step 70/203] elapsed=12.0m eta=22.7m sec/step=10.3 loss=0.6962130546569825 alloc=3.5GB reserved=35.2GB max_alloc=31.9GB
[step 80/203] elapsed=13.8m eta=21.2m sec/step=10.3 loss=0.6954913139343262 alloc=3.5GB reserved=35.2GB max_alloc=31.9GB
[step 90/203] elapsed=15.5m eta=19.4m s